# SFT Adapter Tool-Call Evaluation

This notebook evaluates the SFT LoRA adapter on the same seed eval cases used for the base model.

It loads:
- base model: `HuggingFaceTB/SmolLM-1.7B-Instruct`
- LoRA adapter: `outputs/sft_smollm_20tools`

Then it records a table and saves JSONL/CSV outputs under `outputs/`.

In [1]:
from dataclasses import asdict
from pathlib import Path
import csv
import json
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from toolcall_rl.evaluation.cases import SEED_EVAL_CASES
from toolcall_rl.evaluation.schemas import SYSTEM_PROMPT
from toolcall_rl.evaluation.scoring import score_response

MODEL_ID = "HuggingFaceTB/SmolLM-1.7B-Instruct"
ADAPTER_DIR = PROJECT_ROOT / "outputs" / "sft_smollm_20tools"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

str(ADAPTER_DIR)

'/home/shubeeksh/projects/toolcall-rl/outputs/sft_smollm_20tools'

## Load Model + Adapter

In [2]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.to(device)
model.eval()

device

/home/shubeeksh/projects/toolcall-rl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 218/218 [00:01<00:00, 193.68it/s]


'cuda'

## Generation Helpers

In [3]:
def render_messages(messages):
    if tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    lines = []
    for message in messages:
        lines.append(f"<|{message['role']}|>\n{message['content']}")
    lines.append("<|assistant|>\n")
    return "\n".join(lines)


def generate_response(prompt, max_new_tokens=160):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    rendered = render_messages(messages)
    inputs = tokenizer(rendered, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


generate_response("What is 24 * 17?", max_new_tokens=80)

'{"tool": "calculator", "args": {"expression": "24 * 17"}}'

## Run Evaluation

In [4]:
results = []

for case in SEED_EVAL_CASES:
    response = generate_response(case.prompt)
    score = score_response(response, case)
    results.append(
        {
            "case": asdict(case),
            "response": response,
            "score": asdict(score),
        }
    )

len(results)

20

## Build Table

In [5]:
def flatten_result(index, result):
    case = result["case"]
    score = result["score"]
    return {
        "case_id": index,
        "prompt": case["prompt"],
        "expected_tool": case["expected_tool"],
        "expected_args": json.dumps(case["expected_args"], sort_keys=True),
        "response": result["response"],
        "valid_json": score["valid_json"],
        "json_only": score["json_only"],
        "tool_match": score["tool_match"],
        "args_match": score["args_match"],
        "total_reward": score["total_reward"],
        "parsed": json.dumps(score["parsed"], sort_keys=True),
    }

table_rows = [flatten_result(index, result) for index, result in enumerate(results, start=1)]
table_rows[:1]

[{'case_id': 1,
  'prompt': 'Work out (73 * 9) - 14.',
  'expected_tool': 'calculator',
  'expected_args': '{"expression": "(73 * 9) - 14"}',
  'response': '{"tool": "calculator", "args": {"expression": "73 * 9 - 14"}}',
  'valid_json': 1,
  'json_only': 1,
  'tool_match': 1,
  'args_match': 0,
  'total_reward': 3,
  'parsed': '{"args": {"expression": "73 * 9 - 14"}, "tool": "calculator"}'}]

## Summary

In [6]:
total_cases = len(table_rows)
passed_cases = sum(row["total_reward"] == 4 for row in table_rows)
total_reward = sum(row["total_reward"] for row in table_rows)
max_reward = total_cases * 4

{
    "base_model": MODEL_ID,
    "adapter": str(ADAPTER_DIR),
    "cases": total_cases,
    "passed": passed_cases,
    "total_reward": total_reward,
    "max_reward": max_reward,
}

{'base_model': 'HuggingFaceTB/SmolLM-1.7B-Instruct',
 'adapter': '/home/shubeeksh/projects/toolcall-rl/outputs/sft_smollm_20tools',
 'cases': 20,
 'passed': 15,
 'total_reward': 75,
 'max_reward': 80}

## Results Table

In [7]:
import pandas as pd

df = pd.DataFrame(table_rows)
df[
    [
        "case_id",
        "expected_tool",
        "valid_json",
        "json_only",
        "tool_match",
        "args_match",
        "total_reward",
        "prompt",
        "response",
    ]
]

,case_id,expected_tool,valid_json,json_only,tool_match,args_match,total_reward,prompt,response
0,1,calculator,1,1,1,0,3,Work out (73 * 9) - 14.,"{""tool"": ""calculator"", ""args"": {""expression"": ..."
1,2,google_search,1,1,1,0,3,Search Google for current LoRA adapter merging...,"{""tool"": ""google_search"", ""args"": {""query"": ""L..."
2,3,unit_converter,1,1,1,0,3,I have 27.5 miles; express that in kilometers.,"{""tool"": ""unit_converter"", ""args"": {""value"": 2..."
3,4,text_stats,1,1,1,1,4,"Give text statistics for: ""Adapters are compac...","{""tool"": ""text_stats"", ""args"": {""text"": ""Adapt..."
4,5,string_formatter,1,1,1,1,4,"Convert ""REWARD SIGNAL"" to lowercase.","{""tool"": ""string_formatter"", ""args"": {""text"": ..."
5,6,weather_lookup,1,1,1,1,4,Get the weather for Madrid using celsius units.,"{""tool"": ""weather_lookup"", ""args"": {""city"": ""M..."
6,7,currency_converter,1,1,1,1,4,Exchange 325 USD into CAD.,"{""tool"": ""currency_converter"", ""args"": {""amoun..."
7,8,translate_text,1,1,1,1,4,"Translate ""machine learning"" from English into...","{""tool"": ""translate_text"", ""args"": {""text"": ""m..."
8,9,create_calendar_event,1,1,1,1,4,"Add ""Evaluation review"" to my calendar on 2026...","{""tool"": ""create_calendar_event"", ""args"": {""ti..."
9,10,send_email,1,1,1,1,4,"Email qa@example.com with subject ""Test result...","{""tool"": ""send_email"", ""args"": {""recipient"": ""..."


## Save Results

In [8]:
jsonl_path = OUTPUT_DIR / "sft_20tools_eval_results.jsonl"
csv_path = OUTPUT_DIR / "sft_20tools_eval_results.csv"

with jsonl_path.open("w", encoding="utf-8") as file:
    for result in results:
        file.write(json.dumps(result, ensure_ascii=True) + "\n")

with csv_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=table_rows[0].keys())
    writer.writeheader()
    writer.writerows(table_rows)

{
    "jsonl": str(jsonl_path),
    "csv": str(csv_path),
}

{'jsonl': '/home/shubeeksh/projects/toolcall-rl/outputs/sft_20tools_eval_results.jsonl',
 'csv': '/home/shubeeksh/projects/toolcall-rl/outputs/sft_20tools_eval_results.csv'}